In [3]:
import astropy.units as u
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astropy.time import Time
from astroplan import Observer, FixedTarget
import numpy as np

# Define your observing site
site = EarthLocation(lat=31.9634*u.deg, lon=-111.5992*u.deg, height=2096*u.m)
observer = Observer(location=site, name="Kitt Peak", timezone="US/Arizona")

In [5]:
coord_xmm = [35.5, -5.]
coord_cosmos = [150.1, 2.382]

XMM-LSS

In [7]:
# Define your target
target_ra, target_dec = coord_xmm
target_coord = SkyCoord(ra=target_ra*u.deg, dec=target_dec*u.deg, frame='icrs')
target = FixedTarget(coord=target_coord, name="M1")

# Year to analyze
start_date = Time('2026-07-01 00:00:00')
end_date = Time('2027-06-30 00:00:00')

# Iterate through each night
good_nights = []
current_date = start_date

while current_date < end_date:
    # Get sunset and sunrise for the night
    sunset = observer.sun_set_time(current_date, which='next')
    sunrise = observer.sun_rise_time(sunset, which='next')

    # Sample the night at 5-minute intervals
    night_times = sunset + np.linspace(0, (sunrise - sunset).to(u.hour).value,
                                        int((sunrise - sunset).to(u.hour).value * 12)) * u.hour

    # Calculate sun altitude to apply 15° twilight constraint
    altaz_frame = AltAz(obstime=night_times, location=site)
    sun_coords = get_body("sun", night_times, location=site)
    sun_altaz = sun_coords.transform_to(altaz_frame)

    # Calculate target airmass throughout the night
    target_altaz = target_coord.transform_to(altaz_frame)
    airmass = target_altaz.secz

    # Calculate moon altitude throughout the night
    moon_coords = get_body("moon", night_times, location=site)
    moon_altaz = moon_coords.transform_to(altaz_frame)

    # Count time with all conditions met:
    # - Sun below -15° (15° twilight)
    # - Target airmass < 1.5 and above horizon
    # - Moon below horizon
    good_conditions_mask = (sun_altaz.alt.deg < -15) & \
                          (airmass < 1.5) & (target_altaz.alt.deg > 0) & \
                          (moon_altaz.alt.deg < 0)
    n_good_samples = np.sum(good_conditions_mask)

    # Convert to hours (5-minute sampling)
    hours_observable = n_good_samples * 5.0 / 60.0

    if hours_observable >= 2.0:
        good_airmass_values = airmass[good_conditions_mask]
        good_times = night_times[good_conditions_mask]
        obs_start = observer.astropy_time_to_datetime(good_times[0])
        obs_end = observer.astropy_time_to_datetime(good_times[-1])

        # Convert sunset to local time to get correct local date
        local_sunset = observer.astropy_time_to_datetime(sunset)
        good_nights.append({
            'night': local_sunset.date(),
            'sunset': local_sunset,
            'sunrise': observer.astropy_time_to_datetime(sunrise),
            'obs_start': obs_start,
            'obs_end': obs_end,
            'hours_observable': hours_observable,
            'min_airmass': np.nanmin(good_airmass_values) if n_good_samples > 0 else np.nan,
            'avg_airmass': np.nanmean(good_airmass_values) if n_good_samples > 0 else np.nan
        })

    # Move to next day
    current_date = current_date + 1*u.day

# Print results
print(f"\nNights with ≥2 hours at airmass < 1.5 (15° twilight, moon down) for {target.name}:")
print(f"Total: {len(good_nights)} nights\n")
print(f"{'Night':<12} {'Obs Start':<10} {'Obs End':<10} {'Hours':<8} {'Min AM':<8} {'Avg AM':<8}")
print("-" * 60)

for night in good_nights:
    start_str = night['obs_start'].strftime('%H:%M')
    end_str = night['obs_end'].strftime('%H:%M')
    print(f"{night['night']} {start_str:>9}  {end_str:>8}  {night['hours_observable']:>6.1f}h  {night['min_airmass']:>6.2f}  {night['avg_airmass']:>6.2f}")

# Print summary statistics
if good_nights:
    hours = [n['hours_observable'] for n in good_nights]
    avg_airmasses = [n['avg_airmass'] for n in good_nights]
    print(f"\nSummary:")
    print(f"  Best night: {max(hours):.1f} hours")
    print(f"  Average hours: {np.mean(hours):.1f} hours")
    print(f"  Average airmass: {np.mean(avg_airmasses):.2f}")
    print(f"  Observing season: {good_nights[0]['night']} to {good_nights[-1]['night']}")
else:
    print("\nNo nights meet all criteria.")



Nights with ≥2 hours at airmass < 1.5 (15° twilight, moon down) for M1:
Total: 78 nights

Night        Obs Start  Obs End    Hours    Min AM   Avg AM  
------------------------------------------------------------
2026-08-19     02:46     04:42     2.0h    1.25    1.33
2026-08-20     02:41     04:43     2.1h    1.25    1.33
2026-08-21     02:38     04:44     2.2h    1.25    1.33
2026-08-22     02:33     04:44     2.2h    1.25    1.33
2026-08-23     02:33     04:44     2.2h    1.25    1.32
2026-09-08     01:27     04:29     3.1h    1.25    1.31
2026-09-09     01:24     04:55     3.6h    1.25    1.31
2026-09-10     01:18     04:56     3.7h    1.25    1.31
2026-09-11     01:18     04:56     3.7h    1.25    1.31
2026-09-12     01:10     04:57     3.8h    1.25    1.31
2026-09-13     01:10     04:57     3.8h    1.25    1.31
2026-09-14     01:05     04:58     3.9h    1.25    1.32
2026-09-15     01:01     04:59     4.0h    1.25    1.32
2026-09-16     00:56     04:59     4.1h    1.25    1.32
20

COSMOS

In [6]:
# Define your target
target_ra, target_dec = coord_cosmos
target_coord = SkyCoord(ra=target_ra*u.deg, dec=target_dec*u.deg, frame='icrs')
target = FixedTarget(coord=target_coord, name="M1")

# Year to analyze
start_date = Time('2026-07-01 00:00:00')
end_date = Time('2027-06-30 00:00:00')

# Iterate through each night
good_nights = []
current_date = start_date

while current_date < end_date:
    # Get sunset and sunrise for the night
    sunset = observer.sun_set_time(current_date, which='next')
    sunrise = observer.sun_rise_time(sunset, which='next')

    # Sample the night at 5-minute intervals
    night_times = sunset + np.linspace(0, (sunrise - sunset).to(u.hour).value,
                                        int((sunrise - sunset).to(u.hour).value * 12)) * u.hour

    # Calculate sun altitude to apply 15° twilight constraint
    altaz_frame = AltAz(obstime=night_times, location=site)
    sun_coords = get_body("sun", night_times, location=site)
    sun_altaz = sun_coords.transform_to(altaz_frame)

    # Calculate target airmass throughout the night
    target_altaz = target_coord.transform_to(altaz_frame)
    airmass = target_altaz.secz

    # Calculate moon altitude throughout the night
    moon_coords = get_body("moon", night_times, location=site)
    moon_altaz = moon_coords.transform_to(altaz_frame)

    # Count time with all conditions met:
    # - Sun below -15° (15° twilight)
    # - Target airmass < 1.5 and above horizon
    # - Moon below horizon
    good_conditions_mask = (sun_altaz.alt.deg < -15) & \
                          (airmass < 1.5) & (target_altaz.alt.deg > 0) & \
                          (moon_altaz.alt.deg < 0)
    n_good_samples = np.sum(good_conditions_mask)

    # Convert to hours (5-minute sampling)
    hours_observable = n_good_samples * 5.0 / 60.0

    if hours_observable >= 2.0:
        good_airmass_values = airmass[good_conditions_mask]
        good_times = night_times[good_conditions_mask]
        obs_start = observer.astropy_time_to_datetime(good_times[0])
        obs_end = observer.astropy_time_to_datetime(good_times[-1])

        # Convert sunset to local time to get correct local date
        local_sunset = observer.astropy_time_to_datetime(sunset)
        good_nights.append({
            'night': local_sunset.date(),
            'sunset': local_sunset,
            'sunrise': observer.astropy_time_to_datetime(sunrise),
            'obs_start': obs_start,
            'obs_end': obs_end,
            'hours_observable': hours_observable,
            'min_airmass': np.nanmin(good_airmass_values) if n_good_samples > 0 else np.nan,
            'avg_airmass': np.nanmean(good_airmass_values) if n_good_samples > 0 else np.nan
        })

    # Move to next day
    current_date = current_date + 1*u.day

# Print results
print(f"\nNights with ≥2 hours at airmass < 1.5 (15° twilight, moon down) for {target.name}:")
print(f"Total: {len(good_nights)} nights\n")
print(f"{'Night':<12} {'Obs Start':<10} {'Obs End':<10} {'Hours':<8} {'Min AM':<8} {'Avg AM':<8}")
print("-" * 60)

for night in good_nights:
    start_str = night['obs_start'].strftime('%H:%M')
    end_str = night['obs_end'].strftime('%H:%M')
    print(f"{night['night']} {start_str:>9}  {end_str:>8}  {night['hours_observable']:>6.1f}h  {night['min_airmass']:>6.2f}  {night['avg_airmass']:>6.2f}")

# Print summary statistics
if good_nights:
    hours = [n['hours_observable'] for n in good_nights]
    avg_airmasses = [n['avg_airmass'] for n in good_nights]
    print(f"\nSummary:")
    print(f"  Best night: {max(hours):.1f} hours")
    print(f"  Average hours: {np.mean(hours):.1f} hours")
    print(f"  Average airmass: {np.mean(avg_airmasses):.2f}")
    print(f"  Observing season: {good_nights[0]['night']} to {good_nights[-1]['night']}")
else:
    print("\nNo nights meet all criteria.")



Nights with ≥2 hours at airmass < 1.5 (15° twilight, moon down) for M1:
Total: 91 nights

Night        Obs Start  Obs End    Hours    Min AM   Avg AM  
------------------------------------------------------------
2026-11-19     03:51     05:48     2.0h    1.17    1.29
2026-11-20     03:47     05:48     2.1h    1.17    1.29
2026-12-05     02:50     05:11     2.4h    1.15    1.26
2026-12-06     02:46     05:57     3.2h    1.15    1.24
2026-12-07     02:41     05:58     3.3h    1.15    1.24
2026-12-08     02:37     05:59     3.4h    1.15    1.23
2026-12-09     02:32     05:59     3.5h    1.15    1.23
2026-12-10     02:28     06:00     3.6h    1.15    1.23
2026-12-11     02:23     06:01     3.7h    1.15    1.23
2026-12-12     02:19     06:01     3.8h    1.15    1.23
2026-12-13     02:14     06:02     3.8h    1.15    1.23
2026-12-14     02:15     06:02     3.8h    1.15    1.23
2026-12-15     02:07     06:04     4.0h    1.15    1.23
2026-12-16     02:03     06:04     4.1h    1.15    1.23
20